# Notatnik 0. Wstępny przegląd danych

| | |
|---|---|
| **Aim** | Zobaczyć dane w oryginalnej postaci, zanim cokolwiek zmienimy |
| **Input data** | quaterly files `raw/`, all columbs, 3% of all data - more description of such approach in notebook 1 |
| **Result** | tables in `reports/overview/` |



In [1]:
import polars as pl

import config as C
import functions as F

pl.Config.set_tbl_rows(150)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
OVERVIEW_DIR = C.REPORT_DIR / "overview"
OVERVIEW_DIR.mkdir(parents=True, exist_ok=True)

## Data loading

In [2]:
sizes, columns, codes, masked, programs, previews = [], [], [], [], [], []
CODE_COLUMNS = ["DLQ_STATUS", "Zero_Bal_Code", "MOD_FLAG", "FORBEARANCE_INDICATOR", "ADR_TYPE",
                "PLAN_NUMBER", "PRODUCT", "ORIG_TERM", "PURPOSE", "OCC_STAT", "CHANNEL"]

for path in F.list_raw_files():
    cohort = path.stem
    raw, n_all = F.load_raw_sample(path)
    sizes.append(F.raw_size(raw, cohort, n_all))
    columns.append(F.raw_columns(raw, cohort))
    codes.append(F.raw_codes(raw, cohort, CODE_COLUMNS))
    masked.append(F.masked_balance(raw, cohort))
    programs.append(F.support_programs(raw, cohort))
    previews.append(raw.filter(pl.col("MOD_FLAG").is_not_null()).head(5))
    print(f"{cohort}: {raw.height} rows, {raw.width - 2} columns")

columns = pl.concat(columns)
columns.write_csv(OVERVIEW_DIR / "raw_columns.csv")

2007Q1: 509478 rows, 114 columns


2019Q1: 355368 rows, 114 columns


## 1. Data size

In [3]:
size_table = pl.DataFrame(sizes)
size_table.write_csv(OVERVIEW_DIR / "raw_size.csv")
size_table

cohort,loans_in_file,sampled_loans,sampled_rows,columns,first_month,last_month,median_months_per_loan,max_months_per_loan
"""2007Q1""",253279,7702,509478,114,2007-01-01,2026-06-01,54.0,234
"""2019Q1""",341865,10184,355368,114,2019-01-01,2026-06-01,23.0,90


## 2. Sample Rows


In [4]:
preview_columns = ["cohort", "ACT_PERIOD", "ORIG_DATE", "ORIG_RATE", "ORIG_UPB", "CURRENT_UPB", "LOAN_AGE", "REM_MONTHS",
                   "OLTV", "DTI", "CSCORE_B", "STATE", "DLQ_STATUS", "MOD_FLAG", "Zero_Bal_Code"]
pl.concat(previews).select(preview_columns)

cohort,ACT_PERIOD,ORIG_DATE,ORIG_RATE,ORIG_UPB,CURRENT_UPB,LOAN_AGE,REM_MONTHS,OLTV,DTI,CSCORE_B,STATE,DLQ_STATUS,MOD_FLAG,Zero_Bal_Code
"""2007Q1""","""032007""","""022007""","""6.000""","""116000.00""","""0.00""","""0""","""360""","""80""","""11""","""796""","""GA""","""00""","""N""",null
"""2007Q1""","""042007""","""022007""","""6.000""","""116000.00""","""0.00""","""1""","""359""","""80""","""11""","""796""","""GA""","""00""","""N""",null
"""2007Q1""","""052007""","""022007""","""6.000""","""116000.00""","""0.00""","""2""","""358""","""80""","""11""","""796""","""GA""","""00""","""N""",null
"""2007Q1""","""062007""","""022007""","""6.000""","""116000.00""","""0.00""","""3""","""357""","""80""","""11""","""796""","""GA""","""00""","""N""",null
"""2007Q1""","""072007""","""022007""","""6.000""","""116000.00""","""0.00""","""4""","""356""","""80""","""11""","""796""","""GA""","""00""","""N""",null
"""2019Q1""","""032019""","""032019""","""4.750""","""204000.00""","""0.00""","""-1""","""361""","""78""","""32""","""769""","""IL""","""00""","""N""",null
"""2019Q1""","""042019""","""032019""","""4.750""","""204000.00""","""0.00""","""0""","""360""","""78""","""32""","""769""","""IL""","""00""","""N""",null
"""2019Q1""","""052019""","""032019""","""4.750""","""204000.00""","""0.00""","""1""","""359""","""78""","""32""","""769""","""IL""","""00""","""N""",null
"""2019Q1""","""062019""","""032019""","""4.750""","""204000.00""","""0.00""","""2""","""358""","""78""","""32""","""769""","""IL""","""00""","""N""",null
"""2019Q1""","""072019""","""032019""","""4.750""","""204000.00""","""0.00""","""3""","""357""","""78""","""32""","""769""","""IL""","""00""","""N""",null


## 3. All original Columns

Table just for the 1st cohort.

- `missing_pct`: share of missing rows;
- `n_distinct`: number of different values;
- `changes_over_time`: whether value changes at least once for 1 loan;
- `used_in_project`: if the column is in subsequent analysis;

In [5]:
first_cohort = columns["cohort"][0]
columns.filter(pl.col("cohort") == first_cohort).drop("cohort", "description")

position,column,used_in_project,missing_pct,n_distinct,changes_over_time,top_values,min,max
1,"""POOL_ID""",false,100.0,0,false,"""""",null,null
2,"""LOAN_ID""",true,0.0,7702,false,"""101512638732, 103124784022, 132206726684""",1.0002e11,9.9990e11
3,"""ACT_PERIOD""",true,0.0,234,true,"""032007, 042007, 052007""",12007.0,122025.0
4,"""CHANNEL""",true,0.0,3,true,"""C, R, B""",null,null
5,"""SELLER""",false,0.0,25,true,"""Bank Of America, N.A., Other, Citimortgage, Inc.""",null,null
6,"""SERVICER""",false,1.47,50,true,"""Bank Of America, N.A., Other, Jp Morgan Chase Bank, Na""",null,null
7,"""MASTER_SERVICER""",false,100.0,0,false,"""""",null,null
8,"""ORIG_RATE""",true,0.0,88,false,"""6.250, 5.875, 6.375""",4.5,8.25
9,"""CURR_RATE""",true,1.47,113,true,"""6.250, 5.875, 6.375""",2.0,8.25
10,"""ORIG_UPB""",true,0.0,431,false,"""100000.00, 417000.00, 200000.00""",13000.0,802000.0


### Kolumny całkowicie puste

Pustych kolumn nie da się wykorzystać w tej kohorcie. Kolumna może być pusta w starszej kohorcie, a wypełniona w nowszej.

In [6]:
(columns.group_by("column", "position")
 .agg(empty_in_cohorts=(pl.col("missing_pct") == 100).sum(), cohorts=pl.len())
 .filter(pl.col("empty_in_cohorts") > 0)
 .sort("position"))

column,position,empty_in_cohorts,cohorts
"""POOL_ID""",1,2,2
"""MASTER_SERVICER""",7,2,2
"""ISSUANCE_UPB""",11,2,2
"""FIRST_PAY_IO""",38,2,2
"""MNTHS_TO_AMTZ_IO""",39,2,2
"""MI_CANCEL_FLAG""",43,2,2
"""RPRCH_DTE""",47,2,2
"""CURR_SCHD_PRNCPL""",48,2,2
"""UNSCHD_PRNCPL_CURR""",50,2,2
"""FORECLOSURE_COSTS""",54,1,2


## 4. Braki w kolumnach używanych w projekcie

**Jak czytać:** porównaj kohorty. Duża różnica może oznaczać zmianę sposobu raportowania, a nie zmianę ryzyka.

In [7]:
used = columns.filter(pl.col("used_in_project"))
missing = used.pivot(on="cohort", index=["position", "column", "description"], values="missing_pct").sort("position")
missing.write_csv(OVERVIEW_DIR / "missing_used_columns.csv")
missing

position,column,description,2007Q1,2019Q1
2,"""LOAN_ID""","""loan identifier""",0.0,0.0
3,"""ACT_PERIOD""","""monthly reporting period""",0.0,0.0
4,"""CHANNEL""","""sales channel""",0.0,0.0
8,"""ORIG_RATE""","""interest rate at origination (%)""",0.0,0.0
9,"""CURR_RATE""","""current interest rate (%)""",1.47,2.35
10,"""ORIG_UPB""","""loan amount (USD)""",0.0,0.0
12,"""CURRENT_UPB""","""current balance (USD)""",0.0,0.0
13,"""ORIG_TERM""","""loan term (months)""",0.0,0.0
14,"""ORIG_DATE""","""origination date""",0.0,0.0
16,"""LOAN_AGE""","""loan age (months)""",1.47,2.36


## 5. Wartości najważniejszych kodów

**Jak czytać:** porównaj kody z glosariuszem. `missing` oznacza puste pole.

In [8]:
code_table = pl.concat(codes)
code_table.write_csv(OVERVIEW_DIR / "raw_codes.csv")
for name in CODE_COLUMNS:
    table = (code_table.filter(pl.col("column") == name)
             .sort("cohort", "rows", descending=[False, True])
             .group_by("cohort", maintain_order=True).head(10).drop("column"))
    print(name)
    display(table)

DLQ_STATUS

cohort,value,rows,share_pct
"""2007Q1""","""00""",459718,90.233
"""2007Q1""","""01""",15136,2.971
"""2007Q1""","""XX""",7127,1.399
"""2007Q1""","""02""",5321,1.044
"""2007Q1""","""03""",2823,0.554
"""2007Q1""","""04""",2202,0.432
"""2007Q1""","""05""",1825,0.358
"""2007Q1""","""06""",1507,0.296
"""2007Q1""","""07""",1290,0.253
"""2007Q1""","""08""",1174,0.23


Zero_Bal_Code


cohort,value,rows,share_pct
"""2007Q1""","""missing""",501977,98.528
"""2007Q1""","""01""",6666,1.308
"""2007Q1""","""09""",431,0.085
"""2007Q1""","""03""",177,0.035
"""2007Q1""","""16""",133,0.026
"""2007Q1""","""15""",32,0.006
"""2007Q1""","""02""",31,0.006
"""2007Q1""","""06""",31,0.006
"""2019Q1""","""missing""",346986,97.641
"""2019Q1""","""01""",8335,2.345


MOD_FLAG


cohort,value,rows,share_pct
"""2007Q1""","""N""",456787,89.658
"""2007Q1""","""Y""",45205,8.873
"""2007Q1""","""missing""",7486,1.469
"""2019Q1""","""N""",342859,96.48
"""2019Q1""","""missing""",8367,2.354
"""2019Q1""","""Y""",4142,1.166


... (12 outputs left out)

OCC_STAT


cohort,value,rows,share_pct
"""2007Q1""","""P""",449861,88.298
"""2007Q1""","""I""",39641,7.781
"""2007Q1""","""S""",19976,3.921
"""2019Q1""","""P""",305438,85.95
"""2019Q1""","""I""",32675,9.195
"""2019Q1""","""S""",17255,4.856


CHANNEL


cohort,value,rows,share_pct
"""2007Q1""","""C""",235791,46.281
"""2007Q1""","""R""",194694,38.214
"""2007Q1""","""B""",78993,15.505
"""2019Q1""","""R""",193071,54.33
"""2019Q1""","""C""",128334,36.113
"""2019Q1""","""B""",33963,9.557


## 6. Maskowane saldo

Odsetek wierszy z pustym lub zerowym saldem według wieku kredytu.

**Jak czytać:** wysoki odsetek w pierwszych miesiącach potwierdza maskowanie salda. Wtedy korzystamy z salda z harmonogramu.

In [9]:
masked_table = pl.concat(masked)
masked_table.write_csv(OVERVIEW_DIR / "masked_balance.csv")
masked_table.pivot(on="cohort", index="loan_age", values="masked_pct")

loan_age,2007Q1,2019Q1
0,100.0,100.0
1,100.0,100.0
2,100.0,100.0
3,100.0,100.0
4,100.0,100.0
5,93.7,82.6
6,48.8,27.3
7,10.7,6.4
8,3.4,2.3
9,1.5,0.6


## 7. Modyfikacje, forbearance i odroczenia płatności

To trzy różne zdarzenia:
- **modyfikacja** (`MOD_FLAG`): trwała zmiana warunków kredytu;
- **forbearance** (kod `F`): czasowe zawieszenie lub obniżenie płatności;
- **odroczenie płatności** (`ADR_TYPE`): przeniesienie zaległych rat na późniejszy termin.

**Jak czytać:**
- `first_in_covid_period`: pierwsze zdarzenie przypadło między 03.2020 a 12.2023;
- `delinquent_before_pct`: odsetek kredytów ze statusem opóźnienia w miesiącu przed zdarzeniem.

In [10]:
program_table = pl.concat(programs)
program_table.write_csv(OVERVIEW_DIR / "support_programs.csv")
program_table

cohort,event,loans,share_of_loans_pct,first_in_covid_period,first_outside_covid_period,delinquent_before_pct
"""2007Q1""","""modification""",540,7.01,3,537,99.4
"""2007Q1""","""forbearance""",71,0.92,69,2,40.8
"""2007Q1""","""payment_deferral""",39,0.51,35,4,100.0
"""2019Q1""","""modification""",130,1.28,112,18,100.0
"""2019Q1""","""forbearance""",896,8.8,887,9,14.8
"""2019Q1""","""payment_deferral""",427,4.19,405,22,99.1
